# Explore Event Precipitation and Meteorology

In [1]:
import os
os.chdir('/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/')

# general
import glob
import datetime as dt
from pathlib import Path

# data 
import xarray as xr 
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configure Plotly for Jupyter notebooks
pio.renderers.default = "notebook"
# Alternative renderers you can try if "notebook" doesn't work:
# pio.renderers.default = "plotly_mimetype+notebook"
# pio.renderers.default = "jupyter_lab"

# helper tools
from metpy import calc, units
import scipy.stats as stats
from sklearn.linear_model import LinearRegression

In [2]:
def get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED):
    if WITH_MET and RAW_OR_NORMALIZED == "raw":
        WITH_MET = "_with_raw_met"
    elif WITH_MET and RAW_OR_NORMALIZED == "normalized":
        WITH_MET = "_with_normalized_met"
    else:
        WITH_MET = ""

    if PRODUCT == "gridded":
        PRODUCT_NAME = "_gridded"
        FOLDER_NAME = "gridded_events"
    elif PRODUCT == "events":
        PRODUCT_NAME = ""
        FOLDER_NAME = "events"
    else:
        PRODUCT_NAME = ""
    if SRC in ['asfs', 'sos']:
        SITE = 'kettle_ponds'
    elif SRC in ['bb', 'sail']:
        SITE = 'gothic'
    else:
        SITE = input("Enter site name (gothic or kettle_ponds): ")
    return DATA_DIR / SITE / f"{FOLDER_NAME}{WITH_MET}" / f"{SITE}{PRODUCT_NAME}_precipitation_event_comparisons_{SRC}{WITH_MET}.nc"

In [3]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
OUTPUT_DIR = '/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/04_products/figures/events'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
SRC = "sail" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic/events_with_raw_met/gothic_precipitation_event_comparisons_sail_with_raw_met.nc


In [4]:
gothic_ppt_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/gothic_precipitation_30min.nc')
gothic_met_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/met_30min.nc')
billy_met_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/billy_barr/billy_barr_20211001-20230930_30min.nc')

In [5]:
# set benchmark site 
SITE = 'gothic'  # one of [gothic, kettle_ponds]
BENCHMARK = 'billy_barr_precip'
EVENTS = slice(1,11) # top 10 events
INSTRUMENT = "sail_pluvio"

In [30]:
EVENT = 10  # change to explore different events
event_ds = ds.sel(event_id=EVENT, test_instrument=INSTRUMENT, benchmark=BENCHMARK)


start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values)
event_ppt = gothic_ppt_ds.sel(time=slice(start, end))
event_met = gothic_met_ds.sel(time=slice(start, end))
event_bb_met = billy_met_ds.sel(time=slice(start, end))
# calcualte wind direction
u = event_met['u'].metpy.convert_units('m/s')
v = event_met['v'].metpy.convert_units('m/s')
wind_dir = calc.wind_direction(u, v).metpy.dequantify()
event_met['wind_dir'] = wind_dir
event_ppt_rate = (event_ppt*2)#.rolling(time=2, center=True).mean()  # mm per 30 min to mm per hour
print(event_ds['start_time'].values, event_ds['end_time'].values)

2022-11-03T02:30:00.000000000 2022-11-04T10:00:00.000000000


In [33]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=5, cols=1, 
                    shared_xaxes=True, 
                    subplot_titles=("Precipitation", "Air Temperature", "Wind Speed", "Relative Humidity", "Pressure"),
                    vertical_spacing=0.05,
                    specs=[[{"secondary_y": True}],  # only first subplot
                            [{}], 
                            [{"secondary_y": True}],
                            [{}],
                            [{}]])
# Cumulative precipitation
fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                         y=event_ppt[BENCHMARK].cumsum().values, 
                         name=f'{BENCHMARK.replace("_", " ")} (mm)',
                         mode='lines',
                         line=dict(width=4, color='red')), 
                         row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                         y=event_ppt[INSTRUMENT].cumsum().values, 
                         name=f'{INSTRUMENT.replace("_", " ")} (mm)',
                         mode='lines',
                         line=dict(width=4, color='blue')), 
                         row=1, col=1, )
# Precipitation rate
# secondary y-axis for precipitation rate
fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                         y=event_ppt_rate[BENCHMARK].values, 
                         name=f'{BENCHMARK.replace("_", " ")} Rate (mm/hr)',
                         mode='lines',
                         line=dict(width=2, dash='dot', color='red'),
                         yaxis='y2'),
                         secondary_y=True,
                         row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                         y=event_ppt_rate[INSTRUMENT].values, 
                         name=f'{INSTRUMENT.replace("_", " ")} Rate (mm/hr)',
                         mode='lines',
                         line=dict(width=2, dash='dot', color='blue'),
                         yaxis='y2'),
                        secondary_y=True,
                         row=1, col=1, )
# Temperature
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                         y=event_met['temp_mean'].values, 
                         mode='lines', 
                         name='Air Temperature (°C)',
                         line=dict(width=4,)),
                         row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=5, row=2, col=1)
# Wind speed
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                         y=event_met['wspd_vec_mean'].values, 
                         mode='lines', 
                         name='Wind Speed (m/s)',
                         line=dict(width=4,)), 
                         row=3, col=1)
# Wind direction as secondary y-axis
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                         y=event_met['wind_dir'].values, 
                         mode='markers', 
                         name='Wind Direction (°)',
                         line=dict(width=2, dash='dot', color='gray')),
                         secondary_y=True,
                         row=3, col=1)
fig.add_hline(y=180, line_dash="dash", line_color="black", line_width=5, row=3, col=1, secondary_y=True)
# Plot relative humidity on 4th subplot
fig.add_trace(go.Scatter(x=event_met['time'].values, 
                         y=event_met['rh_mean'].values, 
                         mode='lines', 
                         name='Relative Humidity (%)',
                         line=dict(width=4, color='green')),
                         row=4, col=1)
# Plot pressure on 5th subplot
fig.add_trace(go.Scatter(x=event_met['time'].values,
                            y=event_met['atmos_pressure'].values*10,
                            mode='lines',
                            name='Pressure (hPa)',
                            line=dict(width=4, color='magenta')),
                            row=5, col=1)

fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')

# update yaxis titles
fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr)', overlaying='y', side='right', position=0.15))
fig.update_yaxes(title_text="Wind Speed (m/s)", row=3, col=1)
fig.update_yaxes(title_text="Wind Direction (°)", secondary_y=True, row=3, col=1)
fig.update_yaxes(title_text="Air<br>Temperature (°C)", row=2, col=1)


# Update layout to enable vertical line (spike) on all subplots
for axis in ['xaxis', 'xaxis2', 'xaxis3']:
    fig.layout[axis].update(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='gray',
        spikethickness=1
    )
# move the legend to the bottom with 4 columns
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, traceorder="normal", font=dict(size=18)), 
                  height=1000, width=800,
                  hovermode='x',  # vertical line and combined tooltip
                  )
fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
# decrease vertical white space between each subplot
fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                  title_font_size=24)
fig.update_xaxes(range=[pd.to_datetime(event_bb_met['time'].min().values), pd.to_datetime(event_bb_met['time'].max().values)])
fig.write_image(f'{OUTPUT_DIR}/{SITE}_event_{event_ds["event_id"].values}_exploration_{INSTRUMENT}_vs_{BENCHMARK}.png', scale=2,
                height=1000, width=800)

In [34]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=5, cols=1, 
                    shared_xaxes=True, 
                    subplot_titles=("Precipitation", "Air Temperature", "Wind Speed", "Relative Humidity"),
                    vertical_spacing=0.05,
                    specs=[[{"secondary_y": True}],  # only first subplot
                            [{}], 
                            [{"secondary_y": True}],
                            [{}],
                            [{}]])
# Cumulative precipitation
fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                         y=event_ppt[BENCHMARK].cumsum().values, 
                         name=f'{BENCHMARK.replace("_", " ")} (mm)',
                         mode='lines',
                         line=dict(width=4, color='red')), 
                         row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt['time'].values, 
                         y=event_ppt[INSTRUMENT].cumsum().values, 
                         name=f'{INSTRUMENT.replace("_", " ")} (mm)',
                         mode='lines',
                         line=dict(width=4, color='blue')), 
                         row=1, col=1, )
# Precipitation rate
# secondary y-axis for precipitation rate
fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                         y=event_ppt_rate[BENCHMARK].values, 
                         name=f'{BENCHMARK.replace("_", " ")} Rate (mm/hr)',
                         mode='lines',
                         line=dict(width=2, dash='dot', color='red'),
                         yaxis='y2'),
                         secondary_y=True,
                         row=1, col=1, )
fig.add_trace(go.Scatter(x=event_ppt_rate['time'].values, 
                         y=event_ppt_rate[INSTRUMENT].values, 
                         name=f'{INSTRUMENT.replace("_", " ")} Rate (mm/hr)',
                         mode='lines',
                         line=dict(width=2, dash='dot', color='blue'),
                         yaxis='y2'),
                        secondary_y=True,
                         row=1, col=1, )
# Temperature
fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                         y=event_bb_met['avAirTemp'].values, 
                         mode='lines', 
                         name='Air Temperature (°C)',
                         line=dict(width=4,)),
                         row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=5, row=2, col=1)
# Wind speed
fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                         y=event_bb_met['windSpeed'].values, 
                         mode='lines', 
                         name='Wind Speed (m/s)',
                         line=dict(width=4,)), 
                         row=3, col=1)
# Wind direction as secondary y-axis
fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                         y=event_bb_met['windDirec'].values, 
                         mode='markers', 
                         name='Wind Direction (°)',
                         line=dict(width=2, dash='dot', color='gray')),
                         secondary_y=True,
                         row=3, col=1)
fig.add_hline(y=180, line_dash="dash", line_color="black", line_width=5, row=3, col=1, secondary_y=True)
# Plot relative humidity on 4th subplot
fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                         y=event_bb_met['relHumidty'].values, 
                         mode='lines', 
                         name='Relative Humidity (%)',
                         line=dict(width=4, color='green')),
                         row=4, col=1)
# Plot barometric pressure on 5th subplot
fig.add_trace(go.Scatter(x=event_bb_met['time'].values, 
                         y=event_bb_met['baromPress'].values, 
                         mode='lines', 
                         name='Pressure (hPa)',
                         line=dict(width=4, color='magenta')),
                         row=5, col=1)

fig.update_layout( title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].dt.date.values} to {event_ds["end_time"].dt.date.values}')
fig.layout['yaxis3'].update(showgrid=False)
# update yaxis titles
fig.update_yaxes(title_text="Cumulative<br>Precipitation (mm)", row=1, col=1)
fig.update_layout(yaxis2=dict(title='Precipitation Rate<br>(mm/hr)', overlaying='y', side='right', position=0.15))
fig.update_yaxes(title_text="Wind Speed (m/s)", row=3, col=1)
fig.update_yaxes(title_text="Wind Direction (°)", secondary_y=True, row=3, col=1)
fig.update_yaxes(title_text="Air Temperature (°C)", row=2, col=1)


# Update layout to enable vertical line (spike) on all subplots
for axis in ['xaxis', 'xaxis2', 'xaxis3']:
    fig.layout[axis].update(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='gray',
        spikethickness=1
    )
# move the legend to the bottom with 4 columns
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, traceorder="normal", font=dict(size=18)), 
                  height=1000, width=800,
                  hovermode='x',  # vertical line and combined tooltip
                  )
fig.update_yaxes(showgrid=False, secondary_y=True, row=1, col=1)
fig.update_yaxes(showgrid=False, secondary_y=True, row=3, col=1)
# decrease vertical white space between each subplot
fig.update_layout(margin=dict(t=100, b=100, l=50, r=50), 
                  title_font_size=24)
# update x-axis to limits of the event
fig.update_xaxes(range=[pd.to_datetime(event_bb_met['time'].min().values), pd.to_datetime(event_bb_met['time'].max().values)])
fig.write_image(f'{OUTPUT_DIR}/bb_met/{SITE}_event_{event_ds["event_id"].values}_exploration_{INSTRUMENT}_vs_{BENCHMARK}.png', scale=2,
height=1000, width=800)